# Agentic RAG Pipeline

**Stack**: LangGraph · Groq LLM · Tavily web search · ChromaDB vector store · HuggingFace embeddings

**Flow**:
```
query → embed → retrieve from vector store
      ↘ (low score) → Tavily web search
                   ↓
           LLM synthesis (Groq)
                   ↓
           grounded answer + sources
```

Run all cells once, then `app.py` will import `agent_run` from `agentic_rag_agent.py`
(generated in the last cell).

## 1. Install / Import dependencies

In [ ]:
# Uncomment to install in the notebook kernel
# %pip install -qU langgraph langchain langchain_community langchainhub \
#     langchain_groq langchain_huggingface bs4 tiktoken chromadb \
#     langchain_google_genai tavily-python python-dotenv faiss-cpu


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from project root

GROQ_API_KEY      = os.getenv('GROQ_API_KEY', '')
TAVILY_API_KEY    = os.getenv('TAVILY_API_KEY', '')
GOOGLE_API_KEY    = os.getenv('GOOGLE_API_KEY', '')
LANGCHAIN_API_KEY = os.getenv('LANGCHAIN_API_KEY', '')

# LangSmith tracing — only when key is present
if LANGCHAIN_API_KEY:
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_API_KEY']    = LANGCHAIN_API_KEY
    os.environ['LANGCHAIN_ENDPOINT']   = 'https://api.smith.langchain.com'
    os.environ['LANGCHAIN_PROJECT']    = 'agentic-rag'
    print('LangSmith tracing: ON')
else:
    os.environ['LANGCHAIN_TRACING_V2'] = 'false'
    print('LangSmith tracing: OFF (set LANGCHAIN_API_KEY to enable)')

assert GROQ_API_KEY,   'GROQ_API_KEY is missing — add it to .env'
assert TAVILY_API_KEY, 'TAVILY_API_KEY is missing — add it to .env'
print('Keys loaded OK')


## 2. LLM (Groq) & Embeddings

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

MODEL_NAME = 'llama-3.3-70b-versatile'   # change to any Groq-hosted model

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=1024,
    api_key=GROQ_API_KEY,
)

# Free, local embeddings — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
print('LLM and embeddings ready.')


## 3. Build the Vector Store (ChromaDB)

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
import bs4

# ── Seed documents (swap for your own URLs or local files) ────────────────────
SEED_URLS = [
    'https://lilianweng.github.io/posts/2023-06-23-agent/',
    'https://lilianweng.github.io/posts/2024-02-05-human-data-quality/',
]

loader = WebBaseLoader(
    web_paths=SEED_URLS,
    bs_kwargs=dict(parse_only=bs4.SoupStrainer(
        class_=('post-content', 'post-title', 'post-header')
    )),
)
docs = loader.load()
print(f'Loaded {len(docs)} documents from the web.')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=80
)
chunks = splitter.split_documents(docs)
print(f'Split into {len(chunks)} chunks.')

# Persist to disk so re-runs don't re-download
CHROMA_DIR = './chroma_db'
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)
retriever = vectorstore.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'score_threshold': 0.45, 'k': 4},
)
print('Vector store built and retriever ready.')


## 4. Tools — Retriever + Tavily Web Search

In [ ]:
from langchain.tools.retriever import create_retriever_tool
from langchain_community.tools.tavily_search import TavilySearchResults

retriever_tool = create_retriever_tool(
    retriever,
    name='vector_store_search',
    description=(
        'Search the local knowledge base for information about AI agents, '
        'LLM research, and related topics. Use this FIRST before the web search.'
    ),
)

web_search_tool = TavilySearchResults(
    max_results=4,
    tavily_api_key=TAVILY_API_KEY,
    description=(
        'Search the web for recent or real-time information. '
        'Use when the vector store returns no relevant results.'
    ),
)

tools = [retriever_tool, web_search_tool]
print(f'Tools registered: {[t.name for t in tools]}')


## 5. LangGraph Agent — ReAct Reasoning Loop

In [ ]:
from typing import Annotated, TypedDict, Sequence
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.messages import ToolMessage
from langchain.tools.render import render_text_description
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
import json

# ── State ─────────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert research assistant with access to a vector \
knowledge base and real-time web search.

Strategy:
1. ALWAYS call `vector_store_search` first with the user's query.
2. If the retrieved context is insufficient or outdated, call `tavily_search_results_json`.
3. Synthesise a concise, well-structured answer grounded ONLY in the retrieved context.
4. Cite your sources by mentioning document titles or URLs at the end.
5. Never make up facts; say 'I could not find reliable information' if needed.
"""

# ── Bind tools to LLM ─────────────────────────────────────────────────────────
llm_with_tools = llm.bind_tools(tools)

# ── Nodes ─────────────────────────────────────────────────────────────────────
def agent_node(state: AgentState) -> AgentState:
    """Core ReAct node: thinks, decides which tool to call (or terminates)."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}


tool_node = ToolNode(tools)


def should_continue(state: AgentState) -> str:
    """Route: call tools again, or finish."""
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'
    return END


# ── Graph ──────────────────────────────────────────────────────────────────────
builder = StateGraph(AgentState)
builder.add_node('agent', agent_node)
builder.add_node('tools', tool_node)

builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', should_continue)
builder.add_edge('tools', 'agent')   # always loop back after a tool call

graph = builder.compile()
print('LangGraph agent compiled.')


## 6. Visualise the Graph

In [ ]:
from IPython.display import Image
try:
    Image(graph.get_graph().draw_mermaid_png())
except Exception:
    print(graph.get_graph().draw_mermaid())


## 7. Test the Agent Interactively

In [ ]:
def parse_result(final_state: dict) -> dict:
    """Extract answer, sources, and reasoning steps from the final state."""
    msgs    = final_state.get('messages', [])
    answer  = ''
    sources = []
    steps   = []

    ICONS = {
        'vector_store_search':           ('📚', 'Vector store search'),
        'tavily_search_results_json':    ('🌐', 'Tavily web search'),
    }

    for msg in msgs:
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    icon, label = ICONS.get(tc['name'], ('🔧', tc['name']))
                    arg_preview = str(tc.get('args', ''))[:80]
                    steps.append({'icon': icon, 'text': f'{label}: {arg_preview}'})
            else:
                answer = msg.content
                steps.append({'icon': '✅', 'text': 'Answer synthesised by Groq LLM'})

        elif isinstance(msg, ToolMessage):
            # Try to extract URLs from tool results
            try:
                tool_data = json.loads(msg.content)
                if isinstance(tool_data, list):
                    for item in tool_data:
                        if isinstance(item, dict) and 'url' in item:
                            sources.append({'title': item.get('title', item['url']),
                                            'url': item['url']})
            except Exception:
                # Non-JSON tool result (e.g. retrieved text chunks)
                pass

    return {'answer': answer, 'sources': sources, 'steps': steps}


# Quick test
TEST_QUERY = 'What is an LLM agent and how does it use memory?'

print(f'Query: {TEST_QUERY}\n')
result = graph.invoke({'messages': [HumanMessage(content=TEST_QUERY)]})
parsed = parse_result(result)

print('=== ANSWER ===')
print(parsed['answer'])
print('\n=== SOURCES ===')
for s in parsed['sources']:
    print(f"  - {s['title']}: {s['url']}")
print('\n=== STEPS ===')
for step in parsed['steps']:
    print(f"  {step['icon']} {step['text']}")


## 8. Export `agent_run()` for Streamlit

Running this cell writes `agentic_rag_agent.py` next to `app.py`.  
The Streamlit app auto-imports it and calls `agent_run(query=..., ...)`.  


In [ ]:
AGENT_MODULE = '''
# agentic_rag_agent.py — auto-generated by agentic_rag.ipynb
# Do not edit by hand; re-run the notebook to regenerate.

import os, json
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def agent_run(
    query: str,
    model_name: str = "llama-3.3-70b-versatile",
    max_iterations: int = 5,
    use_web_search: bool = True,
) -> dict:
    """
    Public entry-point called by app.py.

    Returns:
        dict with keys: answer (str), sources (list[dict]), steps (list[dict])
    """
    # Late imports so env vars are already set when this module is imported
    from langchain_groq import ChatGroq
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import Chroma
    from langchain.tools.retriever import create_retriever_tool
    from langchain_community.tools.tavily_search import TavilySearchResults
    from langgraph.graph import StateGraph, START, END
    from langgraph.graph.message import add_messages
    from langgraph.prebuilt import ToolNode
    from langchain_core.messages import SystemMessage, BaseMessage
    from typing import Annotated, TypedDict

    # ── LLM ───────────────────────────────────────────────────────────────────
    llm = ChatGroq(
        model=model_name, temperature=0, max_tokens=1024,
        api_key=os.environ["GROQ_API_KEY"],
    )

    # ── Embeddings + vector store (reuse persisted index) ─────────────────────
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    vectorstore = Chroma(
        persist_directory="./chroma_db",
        embedding_function=embeddings,
    )
    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"score_threshold": 0.45, "k": 4},
    )

    # ── Tools ─────────────────────────────────────────────────────────────────
    active_tools = [
        create_retriever_tool(
            retriever,
            name="vector_store_search",
            description="Search the local knowledge base first.",
        )
    ]
    if use_web_search and os.environ.get("TAVILY_API_KEY"):
        active_tools.append(TavilySearchResults(
            max_results=4,
            tavily_api_key=os.environ["TAVILY_API_KEY"],
        ))

    llm_with_tools = llm.bind_tools(active_tools)

    # ── Graph ──────────────────────────────────────────────────────────────────
    SYSTEM = (
        "You are an expert research assistant. Search the knowledge base first, "
        "then the web if needed. Cite your sources."
    )

    class State(TypedDict):
        messages: Annotated[list[BaseMessage], add_messages]

    iteration = 0

    def agent_node(state):
        nonlocal iteration
        iteration += 1
        msgs = [SystemMessage(content=SYSTEM)] + state["messages"]
        return {"messages": [llm_with_tools.invoke(msgs)]}

    def should_continue(state):
        last = state["messages"][-1]
        if iteration >= max_iterations:
            return END
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return END

    builder = StateGraph(State)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(active_tools))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue)
    builder.add_edge("tools", "agent")
    graph = builder.compile()

    # ── Run ────────────────────────────────────────────────────────────────────
    final = graph.invoke({"messages": [HumanMessage(content=query)]})

    # ── Parse ──────────────────────────────────────────────────────────────────
    answer, sources, steps = "", [], []
    ICONS = {
        "vector_store_search":        ("📚", "Vector store search"),
        "tavily_search_results_json": ("🌐", "Tavily web search"),
    }
    for msg in final.get("messages", []):
        if isinstance(msg, AIMessage):
            if msg.tool_calls:
                for tc in msg.tool_calls:
                    icon, label = ICONS.get(tc["name"], ("🔧", tc["name"]))
                    steps.append({"icon": icon,
                                  "text": f"{label}: {str(tc.get('args',''))[:80]}"})
            else:
                answer = msg.content
                steps.append({"icon": "✅", "text": "Answer synthesised by Groq LLM"})
        elif isinstance(msg, ToolMessage):
            try:
                data = json.loads(msg.content)
                if isinstance(data, list):
                    for item in data:
                        if isinstance(item, dict) and "url" in item:
                            sources.append({"title": item.get("title", item["url"]),
                                            "url": item["url"]})
            except Exception:
                pass

    return {"answer": answer or "No answer generated.",
            "sources": sources,
            "steps": steps}
'''

with open('agentic_rag_agent.py', 'w') as f:
    f.write(AGENT_MODULE.strip())

print('agentic_rag_agent.py written — app.py will now use the real agent.')


## Summary

| Component | Implementation |
|---|---|
| **LLM** | Groq `llama-3.3-70b-versatile` via `langchain-groq` |
| **Embeddings** | `sentence-transformers/all-MiniLM-L6-v2` (local, free) |
| **Vector store** | ChromaDB, persisted to `./chroma_db` |
| **Retrieval** | Similarity-score-threshold retriever (k=4, threshold=0.45) |
| **Web search** | Tavily (`TavilySearchResults`, max 4 results) |
| **Orchestration** | LangGraph ReAct loop: agent → tools → agent → … → END |
| **Tracing** | LangSmith (when `LANGCHAIN_API_KEY` is set) |
| **Frontend** | Streamlit `app.py` imports `agent_run()` from `agentic_rag_agent.py` |
